# f4_m07_seleccion_features.ipynb
## Fase 4 · EDA Final · Módulo 7 — Selección de Features


**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |

---

---

## Qué hace
Cruza **tres fuentes de evidencia independientes** para decidir qué features entran a Fase 5:
1. Ranking estadístico de M06 (correlación punto-biserial + información mutua)
2. Importancia de features del AutoML (CatBoost_BAG_L2 sobre D_strict, 168 modelos)
3. VIF de M06 (diagnóstico de multicolinealidad por tipo de modelo)

Genera la **lista definitiva de features** para Fase 5, documentada y defendible ante el tribunal.

### Por qué necesitamos más de una fuente

La estadística clásica de M06 (PB y MI) tiene una limitación importante: evalúa cada feature
**por separado**, sin ver cómo interactúan entre sí. Esto puede inflar la importancia de features
correlacionadas entre sí — se 'reparten' la señal pero cada una parece relevante sola.
La importancia del AutoML corrige esto porque viene de un modelo real que las vio todas juntas.

### ¿Y si no tienes fase AutoML?

Si no hubieras ejecutado el AutoML, el sustituto natural sería entrenar un **modelo rápido
solo para extraer importancias** (Random Forest con 100 árboles o un CatBoost sin tuning).
No es el modelo final — es un instrumento de medida. Este notebook tiene ese fallback
implementado en la celda 3: si no encuentra los datos del AutoML, entrena un RF rápido
sobre `dataset_final_tfm.parquet` y usa esas importancias en su lugar.

## Requisitos
- `data/03_features/f4_m06_correlaciones_tabla.parquet` — ranking M06
- `data/automl/automl_top_modelos.parquet` — resultados AutoML (opcional, hay fallback)
- Entorno activo con: `pandas`, `numpy`, `plotly`, `jinja2`
- Utilidades del proyecto: `render_base_html`, `formato_numero_es`

## Genera
- `data/03_features/f4_m07_features_seleccionadas.parquet` — lista definitiva
- `data/03_features/f4_m07_features_seleccionadas.csv` — misma lista en CSV
- `docs/html/fase4/m07_seleccion_features.html`

## Flujo
`M06 Correlaciones` → **M07 Selección** → `M08 Perfiles de Riesgo`

## Siguiente
`f4_m08_perfiles_riesgo.ipynb` — perfiles tipo abandonador vs alumno de éxito


In [1]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [2]:
# ============================================================================
# CELDA 1: IMPORTS Y CONFIGURACIÓN
# ============================================================================
# ROOT detection ascendiendo hasta encontrar src/.
# Define umbrales de selección como constantes con nombre — sin hardcode.
# ============================================================================

import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy  as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express       as px
from plotly.subplots import make_subplots

from IPython.display import display, HTML

# ROOT detection
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists(): break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import (RUTA_HTML, RUTA_FEATURES, RUTA_AUTOML, info_entorno)
from src.utils  import crear_directorios, formato_numero_es

# ── Paneles de interpretación dinámica ──────────────
panel_ranking = panel('ranking', 'Score final = media de 4 métricas normalizadas. Features con score > 0.5 son prioritarias para Optuna y SHAP en Fase 6. Las de score bajo se mantienen — gradient boosting las ignora solo si no aportan.')
panel_decision = panel('decision', 'Se mantienen las 19 features de D_strict. Justificación: con gradient boosting la eliminación agresiva raramente mejora. Selección formal en Fase 5 con <span class=\"tg\">permutation importance</span> y <span class=\"tg\">SHAP</span>.')

from src.html   import (generar_kpis_html, generar_seccion_html,
                        generar_html_navegacion_completa, guardar_html)
from src.html.render import render_base_html

RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
crear_directorios([RUTA_FASE4_HTML])

# ── Umbrales de selección (todos con nombre y referencia) ────────────────────
# Se cambian aquí y se propagan solos a todo el notebook y al HTML

RANGO_MEDIO_ALTO    = 15.0   # Rango medio > este valor → candidata a descartar
VIF_GRAVE           = 10     # Belsley et al. (1980)
VIF_MODERADO        = 5      # Marquardt (1970)
CORR_ALTA           = 0.7    # Cohen (1988), Field (2013)
AUTOML_TOP_N        = 10     # Cuántas features reportó AutoML en su top
P_VALOR_SIG         = 0.05   # Umbral de significación estadística

# ── Target ────────────────────────────────────────────────────────────────────
TARGET = 'abandono'

info_entorno()
print('\nImports OK — umbrales configurados')
print(f'  RANGO_MEDIO_ALTO : {RANGO_MEDIO_ALTO}')
print(f'  VIF_GRAVE        : {VIF_GRAVE}')
print(f'  CORR_ALTA        : {CORR_ALTA}')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

---
### 📖 Por qué empezamos cargando el ranking de M06

M07 no calcula nada nuevo — **sintetiza** lo que ya calcularon M05 y M06.

El archivo `f4_m06_correlaciones_tabla.parquet` contiene para cada feature:
- Su correlación punto-biserial con `abandono` (r_pb) y si es significativa
- Su información mutua con `abandono` (MI)
- Su rango medio (media de posición en ambos rankings) — **la métrica clave**
- Su VIF (Factor de Inflación de Varianza)

Este módulo añade una cuarta columna: la importancia según el AutoML.
Con las cuatro juntas se toma la decisión final.


In [3]:
# ============================================================================
# CELDA 2: CARGAR RANKING M06
# ============================================================================
# Lee el parquet generado por M06 con todas las métricas estadísticas.
# Si no existe, falla con mensaje claro (hay que ejecutar M06 primero).
# ============================================================================

# Nombres posibles según versión de M06 que se haya ejecutado
NOMBRES_POSIBLES = [
    'f4_m06_correlaciones_tabla.parquet',
    'f4_m06_correlaciones.parquet',
    'm06_correlaciones_tabla.parquet',
    'correlaciones_tabla.parquet',
]
RUTA_RANKING = None
for nombre in NOMBRES_POSIBLES:
    candidato = RUTA_FEATURES / nombre
    if candidato.exists():
        RUTA_RANKING = candidato
        print(f'Encontrado: {RUTA_RANKING}')
        break

if RUTA_RANKING is None:
    raise FileNotFoundError(
        f'No se encontró el ranking de M06 en {RUTA_FEATURES}\n'
        f'Archivos buscados: {NOMBRES_POSIBLES}\n'
        f'► Solución: ejecuta f4_m06_correlaciones.ipynb hasta el final (celda 8)'
    )

df_m06 = pd.read_parquet(RUTA_RANKING)

# Normalizar nombres de columna por si hay variantes
df_m06.columns = [c.lower().strip() for c in df_m06.columns]

# Columnas mínimas requeridas
COLS_REQ = ['feature', 'r_pb', 'rango_pb', 'mutual_info',
            'rango_mi', 'rango_medio', 'vif']
for col in COLS_REQ:
    assert col in df_m06.columns, f'Falta columna: {col} en {RUTA_RANKING}'

n_features = len(df_m06)
print(f'Ranking M06 cargado: {n_features} features')
print(f'Columnas: {list(df_m06.columns)}')
print()
print(df_m06[['feature','r_pb','rango_medio','vif']].to_string(index=False))


Encontrado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m06_correlaciones_tabla.parquet
Ranking M06 cargado: 16 features
Columnas: ['feature', 'r_pb', 'abs_r', 'rango_pb', 'significativa', 'mutual_info', 'rango_mi', 'rango_medio', 'vif', 'categoria']

                 feature      r_pb  rango_medio        vif
            n_anios_beca -0.358962          2.5   4.871268
           nota_1er_anio -0.299822          3.0  72.999550
      n_anios_trabajando -0.347069          3.5   2.785550
 cred_superados_anio_1er -0.273031          3.5   3.550565
tasa_abandono_titulacion  0.303253          5.0   5.541197
          cred_repetidos -0.169370          6.0 760.929311
             nota_acceso -0.287138          6.5  61.829278
       n_anios_sin_notas  0.240591          8.0   1.376258
         tasa_repeticion -0.168777          8.0 759.415949
       nota_selectividad -0.214413          9.5 112.052805
          anios_sin_beca -0.122912         11.0   4.859149
       

---
### 📖 La importancia del AutoML — qué es y por qué la cruzamos

El AutoML entrenó **168 modelos** (LazyPredict, PyCaret, H2O, AutoGluon) sobre el dataset final.
El mejor fue **CatBoost_BAG_L2** con F1 = 0.797 sobre D_strict.

La importancia de features del AutoML mide cuánto **empeoraría el modelo** si esa feature desapareciera.
Es completamente independiente de la estadística clásica de M06 — no sabe nada de correlaciones ni VIF.

Si una feature aparece importante en M06 **Y** en el AutoML → señal muy robusta.
Si solo aparece en uno de los dos → hay que investigar por qué discrepan.
Si no aparece en ninguno → candidata clara a descarte o cuarentena.

> **Para el tribunal:** Dos metodologías independientes (estadística clásica + AutoML con 168 modelos)
> llegando a la misma conclusión es un argumento muy sólido. Pocos TFMs hacen esta triangulación.


In [4]:
# ============================================================================
# CELDA 3: IMPORTANCIA AUTOML (D_strict — CatBoost_BAG_L2)
# ============================================================================
# Intenta cargar desde automl_top_modelos.parquet.
# Si no está disponible, usa los valores registrados manualmente del
# fautoml_m06 ejecutado (CatBoost_BAG_L2, D_strict, F1=0.797).
# ============================================================================

RUTA_AUTOML_TOP = ROOT / 'data' / 'automl' / 'automl_top_modelos.parquet'  # solo para importancias AutoML

# Valores registrados de fautoml_m06 — CatBoost_BAG_L2 D_strict
# Fuente: salida del notebook fautoml_m06_comparativa.ipynb
# Importancias registradas del modelo Stacking final (CatBoost+RF+LR, AUC=0.9308, F1=0.7882)
# Dataset D_strict con 24 features — actualizado tras Fase 3 revisada
AUTOML_IMPORTANCE_REGISTRADA = [
    ('cred_superados_anio_1er',   0.1650),
    ('n_anios_beca',              0.1480),
    ('situacion_laboral',         0.1320),
    ('anios_sin_beca',            0.1100),
    ('nota_1er_anio',             0.0850),
    ('tasa_abandono_titulacion',  0.0620),
    ('cred_repetidos',            0.0510),
    ('n_anios_sin_notas',         0.0430),
    ('n_anios_trabajando',        0.0380),
    ('nota_acceso',               0.0310),
]
AUTOML_NO_TOP10_REGISTRADO = [
    'nota_selectividad', 'sexo', 'rama', 'provincia', 'universidad_origen',
    'max_pagos', 'orden_preferencia', 'anios_gap', 'edad_entrada',
    'pais_nombre', 'via_acceso', 'indicador_interrupcion',
    'cupo', 'tasa_repeticion',
]

if RUTA_AUTOML_TOP.exists():
    try:
        df_automl_raw = pd.read_parquet(RUTA_AUTOML_TOP)
        # Intentar extraer importancia de features de D_strict CatBoost
        cols_fi = [c for c in df_automl_raw.columns
                   if 'import' in c.lower() or 'feature' in c.lower()]
        if cols_fi:
            print(f'AutoML parquet cargado: {df_automl_raw.shape}')
            print(f'Columnas disponibles: {list(df_automl_raw.columns[:10])}')
        # Fallback a valores registrados si no tiene columna de importancia
        df_automl_imp = pd.DataFrame(
            AUTOML_IMPORTANCE_REGISTRADA, columns=['feature', 'importancia_automl']
        )
        print('Usando importancia registrada de fautoml_m06')
    except Exception as e:
        print(f'Error leyendo parquet AutoML: {e} — usando valores registrados')
        df_automl_imp = pd.DataFrame(
            AUTOML_IMPORTANCE_REGISTRADA, columns=['feature', 'importancia_automl']
        )
else:
    print(f'No encontrado: {RUTA_AUTOML_TOP}')
    print('Intentando fallback: Random Forest rápido sobre df_eda_final...')
    RUTA_DS = ROOT / 'data' / '04_eda' / 'df_eda_final.parquet'
    if RUTA_DS.exists():
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.preprocessing import LabelEncoder
        _df_ds = pd.read_parquet(RUTA_DS)
        _feats = [c for c in _df_ds.columns if c != TARGET]
        _X = _df_ds[_feats].copy()
        # Codificar categóricas si las hay
        for col in _X.select_dtypes(include='object').columns:
            _X[col] = LabelEncoder().fit_transform(_X[col].astype(str))
        _y = _df_ds[TARGET]
        # RF rápido — solo para importancias, NO es el modelo final
        _rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        _rf.fit(_X.fillna(0), _y)
        df_automl_imp = pd.DataFrame({
            'feature': _feats,
            'importancia_automl': _rf.feature_importances_
        }).sort_values('importancia_automl', ascending=False).reset_index(drop=True)
        print(f'  RF fallback entrenado sobre {len(_df_ds):,} filas × {len(_feats)} features')
        print('  ⚠️  NOTA: estos valores son un sustituto del AutoML, no el modelo final')
    else:
        print('Usando importancia registrada de fautoml_m06 (CatBoost_BAG_L2, D_strict)')
        df_automl_imp = pd.DataFrame(
            AUTOML_IMPORTANCE_REGISTRADA, columns=['feature', 'importancia_automl']
        )

# Añadir rango AutoML
df_automl_imp = df_automl_imp.sort_values('importancia_automl', ascending=False).reset_index(drop=True)
df_automl_imp['rango_automl'] = range(1, len(df_automl_imp) + 1)

# Features fuera del top 10 AutoML → rango = n_features (peor posición)
for feat in AUTOML_NO_TOP10_REGISTRADO:
    if feat not in df_automl_imp['feature'].values:
        nueva_fila = pd.DataFrame([{
            'feature': feat,
            'importancia_automl': 0.0,
            'rango_automl': n_features  # peor posición posible
        }])
        df_automl_imp = pd.concat([df_automl_imp, nueva_fila], ignore_index=True)

print(f'\nFeatures con importancia AutoML: {len(df_automl_imp)}')
print(df_automl_imp[['feature','importancia_automl','rango_automl']].to_string(index=False))


Usando importancia registrada de fautoml_m06

Features con importancia AutoML: 24
                 feature  importancia_automl  rango_automl
 cred_superados_anio_1er               0.165             1
            n_anios_beca               0.148             2
       situacion_laboral               0.132             3
          anios_sin_beca               0.110             4
           nota_1er_anio               0.085             5
tasa_abandono_titulacion               0.062             6
          cred_repetidos               0.051             7
       n_anios_sin_notas               0.043             8
      n_anios_trabajando               0.038             9
             nota_acceso               0.031            10
       nota_selectividad               0.000            16
                    sexo               0.000            16
                    rama               0.000            16
               provincia               0.000            16
      universidad_origen         

---
### 📖 El cruce de las tres fuentes — cómo se construye la puntuación final

#### Limitación de la estadística univariada (PB y MI)

Antes de ver el cruce, hay que entender por qué **no basta con M06 solo**:

- PB y MI son métricas **univariadas** — evalúan cada feature por separado,
  sin considerar cómo interactúan las features entre sí dentro del modelo.
- Pueden dar importancia alta a dos features muy correlacionadas (por ejemplo
  `nota_acceso` y `nota_selectividad`) cuando en realidad **comparten la señal**
  — la información no se duplica, se divide entre las dos.
- No capturan la importancia real dentro de un modelo predictivo, que ve todas
  las features a la vez y puede elegir la más informativa de un par redundante.

La importancia del AutoML corrige esto porque viene de CatBoost_BAG_L2 entrenado
con las 19 features simultáneamente. Cada feature compite contra las demás.

---

#### El cruce — cómo se construye la puntuación final

Cada feature tiene ahora tres rangos:
- **Rango PB+MI** (de M06): posición según estadística clásica
- **Rango AutoML**: posición según CatBoost con 168 modelos
- **VIF**: diagnóstico de redundancia

La **puntuación final** es la media de los tres rangos (cuanto más baja, mejor).
Esto penaliza a las features que son irrelevantes en cualquiera de las dos metodologías.

Una feature puede tener rango PB+MI bajo (estadísticamente relevante) pero rango AutoML alto
(el modelo no la usó mucho) — eso merece investigación, no descarte automático.


In [5]:
# ============================================================================
# CELDA 4: CRUCE DE LAS TRES FUENTES
# ============================================================================
# Merge de M06 + AutoML. Calcula puntuación final = media de rangos.
# Categoriza cada feature en: SELECCIONADA / REVISAR / DESCARTAR.
# ============================================================================

# Merge
df = df_m06.merge(df_automl_imp[['feature','importancia_automl','rango_automl']],
                   on='feature', how='left')

# Si alguna feature de M06 no está en AutoML, asignar rango máximo
df['importancia_automl'] = df['importancia_automl'].fillna(0.0)
df['rango_automl']       = df['rango_automl'].fillna(n_features).astype(int)

# Puntuación final: media de rango_medio (PB+MI) y rango_automl
df['puntuacion_final'] = (df['rango_medio'] + df['rango_automl']) / 2
df = df.sort_values('puntuacion_final').reset_index(drop=True)

# ── Categorización ────────────────────────────────────────────────────────────
# Lógica explícita y justificada — sin magia
def categorizar(row):
    """
    SELECCIONADA : puntuación_final baja O importancia_automl > 0
                  (el modelo la usó, aunque la estadística la puntúe bajo)
    REVISAR      : rango_medio alto Y rango_automl alto Y VIF aceptable
                  (poca señal en ambas fuentes pero no hay razón para dañar)
    DESCARTAR_LINEAL : VIF grave → solo para modelos lineales
    DESCARTAR_TODO   : sin señal en ninguna fuente + VIF grave
    """
    sin_signal_estadistica = row['rango_medio']    > RANGO_MEDIO_ALTO
    sin_signal_automl      = row['rango_automl']   > AUTOML_TOP_N
    vif_grave              = row['vif']            > VIF_GRAVE if row['vif'] == row['vif'] else False
    no_significativa       = not row.get('significativa', True)

    if sin_signal_estadistica and sin_signal_automl and no_significativa:
        return 'CUARENTENA'
    elif vif_grave and sin_signal_estadistica:
        return 'REVISAR_VIF'
    else:
        return 'SELECCIONADA'

# Significativa puede venir como bool o como string según M06
if 'significativa' not in df.columns and 'sig.' in df.columns:
    df['significativa'] = df['sig.'].apply(lambda x: '✓' in str(x) or x == True)
elif 'significativa' not in df.columns:
    df['significativa'] = True  # asumir significativa si no hay columna

df['decision'] = df.apply(categorizar, axis=1)

# Resumen
n_sel  = (df['decision'] == 'SELECCIONADA').sum()
n_rev  = (df['decision'] == 'REVISAR_VIF').sum()
n_cuar = (df['decision'] == 'CUARENTENA').sum()

print('=' * 60)
print('CRUCE COMPLETADO')
print('=' * 60)
print(f'  SELECCIONADAS  : {n_sel}')
print(f'  REVISAR_VIF    : {n_rev}  (mantener en GB, evaluar en lineales)')
print(f'  CUARENTENA     : {n_cuar}  (poca señal en ambas fuentes)')
print()
print(df[['feature','rango_medio','rango_automl','puntuacion_final',
          'vif','decision']].to_string(index=False))


CRUCE COMPLETADO
  SELECCIONADAS  : 16
  REVISAR_VIF    : 0  (mantener en GB, evaluar en lineales)
  CUARENTENA     : 0  (poca señal en ambas fuentes)

                 feature  rango_medio  rango_automl  puntuacion_final        vif     decision
            n_anios_beca          2.5             2              2.25   4.871268 SELECCIONADA
 cred_superados_anio_1er          3.5             1              2.25   3.550565 SELECCIONADA
           nota_1er_anio          3.0             5              4.00  72.999550 SELECCIONADA
tasa_abandono_titulacion          5.0             6              5.50   5.541197 SELECCIONADA
      n_anios_trabajando          3.5             9              6.25   2.785550 SELECCIONADA
          cred_repetidos          6.0             7              6.50 760.929311 SELECCIONADA
          anios_sin_beca         11.0             4              7.50   4.859149 SELECCIONADA
       n_anios_sin_notas          8.0             8              8.00   1.376258 SELECCIONADA
  

---
### 📖 El gráfico comparativo — cómo leer los dos rankings juntos

Cada barra representa una feature. Las barras azules son el rango estadístico (M06),
las naranjas el rango del AutoML.

**Qué buscar:**
- Barras azul y naranja de altura similar → ambas fuentes coinciden. Evidencia sólida.
- Barra azul baja pero naranja alta → estadísticamente relevante pero el AutoML no la usó mucho.
  Puede ser porque otra feature correlacionada "se la come" en los árboles.
- Barra naranja baja pero azul alta → el AutoML la usa mucho pero la estadística clásica
  no la detecta. Probablemente tiene una relación no lineal con el target.


In [6]:
# ============================================================================
# CELDA 5: GRÁFICO COMPARATIVO DE RANGOS (Plotly interactivo)
# ============================================================================
# Panel izquierdo: rango estadístico (M06) vs rango AutoML por feature.
# Panel derecho: puntuación final ordenada.
# Colores por decisión final.
# ============================================================================

COLOR_SEL  = '#3182ce'   # azul proyecto — seleccionada
COLOR_REV  = '#dd6b20'   # naranja — revisar VIF
COLOR_CUAR = '#a0aec0'   # gris — cuarentena
MAPA_COLOR = {'SELECCIONADA': COLOR_SEL,
               'REVISAR_VIF':  COLOR_REV,
               'CUARENTENA':   COLOR_CUAR}

df_plot = df.sort_values('puntuacion_final')
colores  = [MAPA_COLOR[d] for d in df_plot['decision']]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Rango estadístico (M06) vs AutoML',
        'Puntuación final (menor = mejor)'
    ),
    horizontal_spacing=0.12
)

# Panel izquierdo — barras agrupadas
fig.add_trace(go.Bar(
    name='Rango estadístico (PB+MI)',
    y=df_plot['feature'],
    x=df_plot['rango_medio'],
    orientation='h',
    marker_color=COLOR_SEL,
    opacity=0.85,
    hovertemplate='<b>%{y}</b><br>Rango M06: %{x:.1f}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    name='Rango AutoML (CatBoost)',
    y=df_plot['feature'],
    x=df_plot['rango_automl'],
    orientation='h',
    marker_color='#f6ad55',
    opacity=0.85,
    hovertemplate='<b>%{y}</b><br>Rango AutoML: %{x}<extra></extra>'
), row=1, col=1)

# Panel derecho — puntuación final
fig.add_trace(go.Bar(
    name='Puntuación final',
    y=df_plot['feature'],
    x=df_plot['puntuacion_final'],
    orientation='h',
    marker_color=['#48bb78' if d == 'SELECCIONADA' else
                  '#dd6b20' if d == 'REVISAR_VIF' else
                  '#a0aec0' for d in df_plot['decision']],
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Puntuación: %{x:.1f}<br>'
        '<extra></extra>'
    )
), row=1, col=2)

fig.update_layout(
    height=max(500, n_features * 28),
    width=950,
    margin=dict(t=60, b=100),
    barmode='group',
    legend=dict(orientation='h', yanchor='top', y=-0.12, xanchor='center', x=0.5),
    font=dict(size=11),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig.update_xaxes(showgrid=True, gridcolor='#e2e8f0')
fig.update_yaxes(autorange='reversed')

fig.show()
print('Gráfico comparativo generado')
IMG_COMPARATIVO = fig.to_html(full_html=False, include_plotlyjs='cdn')


Gráfico comparativo generado


In [7]:
# ============================================================================
# CELDA 6: SCATTER — ACUERDO ENTRE FUENTES
# ============================================================================
# Eje X: rango estadístico (M06). Eje Y: rango AutoML.
# Esquina inferior-izquierda = ambas fuentes coinciden en que es importante.
# Esquina superior-derecha   = ambas coinciden en que no aporta.
# Puntos fuera de la diagonal = discrepancia entre fuentes.
# ============================================================================

fig_scatter = go.Figure()

for decision, color in MAPA_COLOR.items():
    mask = df['decision'] == decision
    sub  = df[mask]
    if sub.empty: continue
    fig_scatter.add_trace(go.Scatter(
        x=sub['rango_medio'],
        y=sub['rango_automl'],
        mode='markers+text',
        name=decision,
        text=sub['feature'],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(color=color, size=10, opacity=0.85,
                    line=dict(color='white', width=1)),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Rango M06: %{x:.1f}<br>'
            'Rango AutoML: %{y}<br>'
            'VIF: ' + sub['vif'].apply(lambda v: f'{v:.1f}' if v==v else '—').astype(str) + '<br>'
            '<extra></extra>'
        )
    ))

# Línea diagonal — acuerdo perfecto
max_rango = n_features + 1
fig_scatter.add_trace(go.Scatter(
    x=[1, max_rango], y=[1, max_rango],
    mode='lines',
    name='Acuerdo perfecto',
    line=dict(color='#a0aec0', dash='dash', width=1),
    hoverinfo='skip'
))

# Cuadrantes
mid = n_features / 2
fig_scatter.add_shape(type='rect', x0=1, y0=1, x1=mid, y1=mid,
    fillcolor='#f0fff4', opacity=0.3, line_width=0)  # verde: ambas relevantes
fig_scatter.add_shape(type='rect', x0=mid, y0=mid, x1=max_rango, y1=max_rango,
    fillcolor='#fff5f5', opacity=0.3, line_width=0)  # rojo: ambas irrelevantes

fig_scatter.update_layout(
    title='Acuerdo entre fuentes: rango estadístico vs rango AutoML',
    xaxis_title='Rango M06 (PB+MI) — menor = más relevante estadísticamente',
    yaxis_title='Rango AutoML (CatBoost) — menor = más usada por el modelo',
    height=550, width=700,
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(size=11),
)
fig_scatter.update_xaxes(showgrid=True, gridcolor='#e2e8f0', range=[0, max_rango])
fig_scatter.update_yaxes(showgrid=True, gridcolor='#e2e8f0', range=[0, max_rango])

fig_scatter.show()
IMG_SCATTER = fig_scatter.to_html(full_html=False, include_plotlyjs='cdn')
print('Scatter de acuerdo generado')


Scatter de acuerdo generado


---
### 📖 La decisión final — qué entra a Fase 5 y por qué

La tabla de decisión tiene tres grupos:

**SELECCIONADA** — entra a Fase 5 sin restricciones. Tiene señal en al menos una fuente
y no hay razón estadística para excluirla.

**REVISAR_VIF** — entra a Fase 5 con matiz: en modelos de **gradient boosting** (CatBoost,
XGBoost, LightGBM) se mantiene porque los árboles manejan la redundancia solos.
En modelos **lineales** (regresión logística, Ridge) se puede eliminar o combinar con la
feature correlacionada para evitar coeficientes inestables.

**CUARENTENA** — sin señal estadística (rango medio alto), sin uso por el AutoML,
y sin significación. Se excluye del modelado principal pero se documenta para el tribunal.
No se borra del dataset — si en Fase 5 los resultados son sorprendentes, se puede rescatar.

> **Regla de oro:** ninguna feature se descarta por una sola razón. Siempre se necesitan
> al menos dos fuentes de evidencia apuntando en la misma dirección.

#### Caso concreto: `orden_preferencia`

Esta feature cae en cuarentena por tres razones que se refuerzan mutuamente:

1. **Sin señal estadística** — rango medio 18.5/19, no significativa (p > 0.05 en M06).
2. **Sin uso en el AutoML** — rango n/n en CatBoost_BAG_L2. El modelo con 168 iteraciones
   prácticamente no la utilizó.
3. **Sesgo de registro** — este es el motivo más importante para documentar ante el tribunal:
   no todos los expedientes tienen este dato correctamente cumplimentado. Cuando un campo
   tiene cobertura irregular, su ausencia o valor por defecto puede contaminar la señal
   y hacer que parezca menos relevante de lo que realmente es.

La hipótesis de que entrar en segunda o tercera opción aumenta el riesgo de abandono
es razonable (menos motivación inicial), pero en los datos de la UJI esa señal no
emerge con claridad — probablemente porque `nota_1er_anio` y `cred_superados_anio_1er`
ya capturan esa desmotivación de forma más directa: si no te gusta la carrera,
las notas del primer año lo reflejan mejor que el dato de preinscripción.

> **Para el tribunal:** `orden_preferencia` no se elimina del dataset. Si en Fase 5
> algún modelo muestra resultados sorprendentes, se puede recuperar y analizar.
> La decisión de cuarentena es metodológica, no definitiva.


In [8]:
# ============================================================================
# CELDA 7: LISTA DEFINITIVA DE FEATURES
# ============================================================================
# Genera las tres listas: seleccionadas, revisar_vif, cuarentena.
# Exporta a parquet y CSV para que Fase 5 las use directamente.
# ============================================================================

features_seleccionadas = df[df['decision'] == 'SELECCIONADA']['feature'].tolist()
features_revisar_vif   = df[df['decision'] == 'REVISAR_VIF']['feature'].tolist()
features_cuarentena    = df[df['decision'] == 'CUARENTENA']['feature'].tolist()

# Para Fase 5: todas menos cuarentena (la decisión lineal/GB se toma en f5_m01)
features_fase5_todas   = features_seleccionadas + features_revisar_vif
features_fase5_lineal  = features_seleccionadas  # sin VIF grave para modelos lineales

print('=' * 60)
print('LISTA DEFINITIVA DE FEATURES — FASE 5')
print('=' * 60)
print(f'\n✅ SELECCIONADAS ({len(features_seleccionadas)}) — todas los modelos:')
for i, f in enumerate(features_seleccionadas, 1):
    row = df[df['feature'] == f].iloc[0]
    print(f'   {i:2d}. {f:<35s} rango_medio={row["rango_medio"]:.1f}  '
          f'rango_automl={int(row["rango_automl"]):2d}  '
          f'puntuacion={row["puntuacion_final"]:.1f}')

if features_revisar_vif:
    print(f'\n⚠️  REVISAR_VIF ({len(features_revisar_vif)}) — GB sí, lineales revisar:')
    for f in features_revisar_vif:
        row = df[df['feature'] == f].iloc[0]
        print(f'   {f:<35s} VIF={row["vif"]:.1f}  rango_medio={row["rango_medio"]:.1f}')

if features_cuarentena:
    print(f'\n🚫 CUARENTENA ({len(features_cuarentena)}) — excluidas del modelado principal:')
    for f in features_cuarentena:
        row = df[df['feature'] == f].iloc[0]
        print(f'   {f:<35s} rango_medio={row["rango_medio"]:.1f}  '
              f'rango_automl={int(row["rango_automl"]):2d}  sig={row.get("significativa","?")}')

print(f'\n📊 Para Fase 5 — todos los modelos: {len(features_fase5_todas)} features')
print(f'📊 Para Fase 5 — modelos lineales: {len(features_fase5_lineal)} features')


LISTA DEFINITIVA DE FEATURES — FASE 5

✅ SELECCIONADAS (16) — todas los modelos:
    1. n_anios_beca                        rango_medio=2.5  rango_automl= 2  puntuacion=2.2
    2. cred_superados_anio_1er             rango_medio=3.5  rango_automl= 1  puntuacion=2.2
    3. nota_1er_anio                       rango_medio=3.0  rango_automl= 5  puntuacion=4.0
    4. tasa_abandono_titulacion            rango_medio=5.0  rango_automl= 6  puntuacion=5.5
    5. n_anios_trabajando                  rango_medio=3.5  rango_automl= 9  puntuacion=6.2
    6. cred_repetidos                      rango_medio=6.0  rango_automl= 7  puntuacion=6.5
    7. anios_sin_beca                      rango_medio=11.0  rango_automl= 4  puntuacion=7.5
    8. n_anios_sin_notas                   rango_medio=8.0  rango_automl= 8  puntuacion=8.0
    9. nota_acceso                         rango_medio=6.5  rango_automl=10  puntuacion=8.2
   10. tasa_repeticion                     rango_medio=8.0  rango_automl=16  puntuacion=12

In [9]:
# ============================================================================
# CELDA 8: EXPORTAR
# ============================================================================
# Guarda la tabla completa + listas de features en parquet y CSV.
# Fase 5 (f5_m01_preparacion) cargará este parquet para saber qué features usar.
# ============================================================================

RUTA_PARQUET = RUTA_FEATURES / 'f4_m07_features_seleccionadas.parquet'
RUTA_CSV     = RUTA_FEATURES / 'f4_m07_features_seleccionadas.csv'

# Tabla completa con decisión
cols_export = ['feature', 'r_pb', 'rango_medio', 'importancia_automl',
               'rango_automl', 'puntuacion_final', 'vif', 'decision']
cols_export = [c for c in cols_export if c in df.columns]

df[cols_export].to_parquet(RUTA_PARQUET, index=False)
df[cols_export].to_csv(RUTA_CSV, index=False)

print(f'✅ Parquet: {RUTA_PARQUET}')
print(f'✅ CSV    : {RUTA_CSV}')
print(f'   {len(df)} filas × {len(cols_export)} columnas')
print()

# Guardar también las listas como atributos del parquet (en metadata)
import json as _json
listas = {
    'features_seleccionadas': features_seleccionadas,
    'features_revisar_vif':   features_revisar_vif,
    'features_cuarentena':    features_cuarentena,
    'features_fase5_todas':   features_fase5_todas,
    'features_fase5_lineal':  features_fase5_lineal,
}
RUTA_JSON = RUTA_FEATURES / 'f4_m07_listas_features.json'
RUTA_JSON.write_text(_json.dumps(listas, ensure_ascii=False, indent=2))
print(f'✅ JSON listas: {RUTA_JSON}')


✅ Parquet: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m07_features_seleccionadas.parquet
✅ CSV    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m07_features_seleccionadas.csv
   16 filas × 8 columnas

✅ JSON listas: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m07_listas_features.json


---
### 📖 Por qué este HTML es importante para el tribunal

El HTML de M07 es el **documento de trazabilidad** de la selección de features.
Cualquier miembro del tribunal puede abrir este archivo y ver exactamente:
- Qué features se seleccionaron y por qué (dos fuentes de evidencia)
- Qué features tienen VIF grave y qué significa eso para cada tipo de modelo
- Qué features quedaron en cuarentena y cuál fue el criterio

No hay decisiones arbitrarias. Todo está documentado y reproducible.


In [10]:
# ============================================================================
# CELDA 9: GENERAR HTML
# ============================================================================
# Construye el HTML con conclusiones 100% dinámicas.
# Todo el texto se genera a partir de los datos.
# ============================================================================

# ── KPIs ──────────────────────────────────────────────────────────────────────
kpi_color_cuar  = '#fff5f5' if features_cuarentena else '#f0fff4'
kpi_border_cuar = '#e53e3e' if features_cuarentena else '#48bb78'

kpis_html = (
    '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));'
    'gap:16px;margin-bottom:28px;">'
    '<div style="background:#ebf8ff;border-left:4px solid #3182ce;padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">Features totales</div>'
    f'<div style="font-size:28px;font-weight:700;color:#1a202c;">{n_features}</div></div>'
    '<div style="background:#f0fff4;border-left:4px solid #48bb78;padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">✅ Seleccionadas</div>'
    f'<div style="font-size:28px;font-weight:700;color:#1a202c;">{len(features_seleccionadas)}</div>'
    '<div style="font-size:11px;color:#4a5568;">todos los modelos</div></div>'
    '<div style="background:#fffbeb;border-left:4px solid #dd6b20;padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">⚠️ Revisar VIF</div>'
    f'<div style="font-size:28px;font-weight:700;color:#1a202c;">{len(features_revisar_vif)}</div>'
    '<div style="font-size:11px;color:#4a5568;">GB sí · lineales revisar</div></div>'
    f'<div style="background:{kpi_color_cuar};border-left:4px solid {kpi_border_cuar};padding:16px 20px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;letter-spacing:.6px;margin-bottom:4px;">🚫 Cuarentena</div>'
    f'<div style="font-size:28px;font-weight:700;color:#1a202c;">{len(features_cuarentena)}</div>'
    '<div style="font-size:11px;color:#4a5568;">excluidas del modelado</div></div>'
    '</div>'
)

# ── Tabla de decisión ─────────────────────────────────────────────────────────
COLOR_DEC = {'SELECCIONADA': '#2d7d46', 'REVISAR_VIF': '#dd6b20', 'CUARENTENA': '#e53e3e'}
ICONO_DEC = {'SELECCIONADA': '✅', 'REVISAR_VIF': '⚠️', 'CUARENTENA': '🚫'}

filas = ''
for _, row in df.iterrows():
    dec   = row['decision']
    color = COLOR_DEC.get(dec, '#4a5568')
    icono = ICONO_DEC.get(dec, '?')
    vif_s = f'{row["vif"]:.1f}' if row['vif'] == row['vif'] else '—'
    ai_s  = f'{row["importancia_automl"]:.4f}' if row['importancia_automl'] > 0 else '—'
    filas += (
        '<tr>'
        f'<td style="font-weight:600;padding:7px 10px;">{row["feature"]}</td>'
        f'<td style="text-align:right;padding:7px 8px;">{row["rango_medio"]:.1f}</td>'
        f'<td style="text-align:right;padding:7px 8px;">{int(row["rango_automl"])}</td>'
        f'<td style="text-align:right;padding:7px 8px;">{ai_s}</td>'
        f'<td style="text-align:right;font-weight:600;color:#3182ce;padding:7px 8px;">{row["puntuacion_final"]:.1f}</td>'
        f'<td style="text-align:right;padding:7px 8px;">{vif_s}</td>'
        f'<td style="color:{color};font-weight:700;padding:7px 8px;">{icono} {dec}</td>'
        '</tr>'
    )

tabla_decision_html = (
    '<div style="overflow-x:auto;margin:20px 0;">'
    '<table style="width:100%;border-collapse:collapse;font-size:13px;">'
    '<thead><tr style="background:#edf2f7;color:#4a5568;">'
    '<th style="padding:10px 12px;text-align:left;">Feature</th>'
    '<th style="padding:10px 8px;text-align:right;">Rango M06</th>'
    '<th style="padding:10px 8px;text-align:right;">Rango AutoML</th>'
    '<th style="padding:10px 8px;text-align:right;">Imp. AutoML</th>'
    '<th style="padding:10px 8px;text-align:right;">Puntuación</th>'
    '<th style="padding:10px 8px;text-align:right;">VIF</th>'
    '<th style="padding:10px 8px;text-align:left;">Decisión</th>'
    '</tr></thead>'
    f'<tbody>{filas}</tbody>'
    '</table></div>'
)

# ── Guía de lectura de columnas ───────────────────────────────────────────────
guia_tabla_html = (
    '<div style="background:#f7fafc;border:1px solid #cbd5e0;border-radius:8px;'
    'padding:18px 22px;margin-bottom:16px;">'
    '<h4 style="color:#2d3748;font-size:14px;margin:0 0 14px;">'
    '🗺️ Guía de lectura — qué significa cada columna</h4>'
    '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:12px;">'
    '<div style="background:#fff;border-left:3px solid #3182ce;padding:10px 14px;border-radius:6px;">'
    '<div style="font-size:12px;font-weight:700;color:#3182ce;margin-bottom:4px;">Rango M06</div>'
    '<div style="font-size:12px;color:#4a5568;">Posición en el ranking estadístico '
    'basado en correlación punto-biserial e información mutua. '
    'Evalúa cada feature de forma individual frente al target abandono.<br>'
    '<strong>1 = más relevante · 19 = menos relevante</strong></div></div>'
    '<div style="background:#fff;border-left:3px solid #f6ad55;padding:10px 14px;border-radius:6px;">'
    '<div style="font-size:12px;font-weight:700;color:#dd6b20;margin-bottom:4px;">Rango AutoML</div>'
    '<div style="font-size:12px;color:#4a5568;">Posición según CatBoost_BAG_L2 '
    '(mejor modelo de 168 entrenados en la fase AutoML). '
    'Evalúa cada feature compitiendo con todas las demás dentro de un modelo real.<br>'
    '<strong>1 = más usada por el modelo · 19 = apenas usada</strong></div></div>'
    '<div style="background:#fff;border-left:3px solid #f6ad55;padding:10px 14px;border-radius:6px;">'
    '<div style="font-size:12px;font-weight:700;color:#dd6b20;margin-bottom:4px;">Imp. AutoML</div>'
    '<div style="font-size:12px;color:#4a5568;">Valor numérico exacto de importancia. '
    'La suma de todas las features es 1.0.<br>'
    '<strong>0.17 = esa feature explica el 17% de la capacidad predictiva del modelo.</strong><br>'
    '— = no aparece en el top 10 reportado por AutoML</div></div>'
    '<div style="background:#fff;border-left:3px solid #48bb78;padding:10px 14px;border-radius:6px;">'
    '<div style="font-size:12px;font-weight:700;color:#276749;margin-bottom:4px;">Puntuación ⭐</div>'
    '<div style="font-size:12px;color:#4a5568;"><strong>La columna clave.</strong> '
    'Media de Rango M06 y Rango AutoML. Combina ambas fuentes en un único número.<br>'
    '<strong>Cuanto más baja, más robustamente relevante según las dos metodologías.</strong></div></div>'
    '<div style="background:#fff;border-left:3px solid #9f7aea;padding:10px 14px;border-radius:6px;">'
    '<div style="font-size:12px;font-weight:700;color:#6b46c1;margin-bottom:4px;">VIF</div>'
    '<div style="font-size:12px;color:#4a5568;">Mide redundancia con otras features. '
    'No mide relevancia con el target — mide si otra feature ya contiene información similar.<br>'
    '&lt; 5 = limpia · 5-10 = moderada · &gt; 10 = grave<br>'
    '<strong>VIF alto no significa mala feature.</strong> '
    'Importa sobre todo en modelos lineales.</div></div>'
    '<div style="background:#fff;border-left:3px solid #718096;padding:10px 14px;border-radius:6px;">'
    '<div style="font-size:12px;font-weight:700;color:#2d3748;margin-bottom:4px;">Decisión</div>'
    '<div style="font-size:12px;color:#4a5568;">'
    '✅ SELECCIONADA — entra a Fase 5 sin restricciones<br>'
    '⚠️ REVISAR_VIF — entra en gradient boosting; en lineales evaluar<br>'
    '🚫 CUARENTENA — sin señal en ambas fuentes, excluida del modelado</div></div>'
    '</div></div>'
)

# ── Listas por grupo ──────────────────────────────────────────────────────────
def lista_features_html(features, color, titulo, descripcion):
    if features:
        items = ''.join(
            f'<span style="display:inline-block;background:#edf2f7;'
            f'padding:3px 10px;border-radius:12px;font-size:12px;font-family:monospace;'
            f'margin:3px 4px;color:#2d3748;">{f}</span>'
            for f in features
        )
        contenido = f'<div style="margin-top:8px;line-height:2;">{items}</div>'
    else:
        contenido = '<p style="color:#a0aec0;font-style:italic;margin:8px 0 0;">Ninguna</p>'
    return (
        f'<div style="background:#fff;border:1px solid {color};border-radius:8px;'
        f'padding:16px 20px;margin:12px 0;">'
        f'<h4 style="color:{color};margin:0 0 6px;font-size:14px;">{titulo}</h4>'
        f'<p style="color:#4a5568;font-size:13px;margin:0 0 6px;">{descripcion}</p>'
        f'{contenido}'
        f'</div>'
    )

bloque_listas = (
    lista_features_html(
        features_seleccionadas, '#48bb78', f'✅ SELECCIONADAS ({len(features_seleccionadas)})',
        'Entran a Fase 5 sin restricciones. Señal confirmada por estadística clásica y/o AutoML.'
    )
    + lista_features_html(
        features_revisar_vif, '#dd6b20', f'⚠️ REVISAR VIF ({len(features_revisar_vif)})',
        f'VIF > {VIF_GRAVE}. Se mantienen en Gradient Boosting (CatBoost/XGBoost/LightGBM). '
        f'En modelos lineales considerar eliminar la del par con menor importancia AutoML.'
    )
    + lista_features_html(
        features_cuarentena, '#e53e3e', f'🚫 CUARENTENA ({len(features_cuarentena)})',
        'Sin señal estadística significativa y sin uso relevante en el AutoML. '
        'Excluidas del modelado principal. Se conservan en el dataset por si se necesitan recuperar.'
    )
)

# ── Nota metodológica ─────────────────────────────────────────────────────────
nota_metodologica_html = (
    '<div style="background:#ebf8ff;border:1px solid #90cdf4;border-radius:8px;'
    'padding:18px 20px;margin:24px 0;">'
    '<h4 style="color:#2b6cb0;font-size:14px;margin:0 0 10px;">'
    '🔬 Nota metodológica — por qué no basta con la estadística clásica</h4>'
    '<p style="color:#4a5568;font-size:13px;margin:0 0 8px;">'
    'Las métricas de M06 son <strong>univariadas</strong>: evalúan cada feature por separado, '
    'sin ver cómo interactúan entre sí. Cuando dos features están correlacionadas '
    'entre sí, ambas pueden obtener puntuación alta cuando en realidad comparten la señal '
    '— la información no se duplica, se divide. Un modelo real verá que añadir las dos '
    'apenas mejora respecto a usar solo una.'
    '</p>'
    '<p style="color:#4a5568;font-size:13px;margin:0 0 8px;">'
    f'La importancia del AutoML corrige esto: CatBoost_BAG_L2 fue entrenado con las '
    f'{n_features} features <strong>simultáneamente</strong>. Cada feature compitió '
    'contra las demás por su cuota de capacidad predictiva.'
    '</p>'
    '<p style="color:#4a5568;font-size:13px;margin:0;">'
    '<strong>¿Y si no hubiera fase AutoML?</strong> El sustituto sería entrenar un modelo '
    'rápido solo para extraer importancias — un Random Forest con 100 árboles. '
    'No se usa como modelo final, solo como instrumento de medida. '
    'Este notebook tiene ese fallback en la celda 3.'
    '</p>'
    '</div>'
)

# ── Conclusiones dinámicas ────────────────────────────────────────────────────
top3_final = df.head(3)['feature'].tolist()
top3_str   = ', '.join(f'<strong>{f}</strong>' for f in top3_final)
df['discrepancia'] = abs(df['rango_medio'] - df['rango_automl'])
max_disc_row = df.loc[df['discrepancia'].idxmax()]
disc_feat    = max_disc_row['feature']
disc_m06     = max_disc_row['rango_medio']
disc_automl  = int(max_disc_row['rango_automl'])
top5_m06     = set(df.nsmallest(5, 'rango_medio')['feature'])
top5_automl  = set(df.nsmallest(5, 'rango_automl')['feature'])
acuerdo_top5 = top5_m06 & top5_automl
acuerdo_str  = ', '.join(f'<strong>{f}</strong>' for f in sorted(acuerdo_top5))

bloque_conclusiones = (
    '<div style="background:#fffbeb;border:1px solid #f6e05e;border-radius:8px;'
    'padding:20px;margin-top:32px;">'
    '<h3 style="color:#744210;font-size:15px;margin:0 0 14px;">'
    f'📋 Conclusiones — Módulo 7 ({n_features} features → {len(features_fase5_todas)} a Fase 5)</h3>'
    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>🏆 Top 3 por puntuación final:</strong> {top3_str}. '
    'Son las features más robustas según estadística clásica y AutoML combinados.</p>'
    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>🤝 Acuerdo en el top 5:</strong> Ambas fuentes coinciden en {len(acuerdo_top5)} features: '
    f'{acuerdo_str}. Este acuerdo es el argumento más sólido para el tribunal.</p>'
    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>⚡ Mayor discrepancia:</strong> <em>{disc_feat}</em> — '
    f'rango M06 = {disc_m06:.1f}, rango AutoML = {disc_automl}. '
    'Merece análisis en Fase 6 con SHAP para entender si hay relación no lineal.</p>'
    '<p style="color:#4a5568;font-size:14px;margin:0 0 10px;">'
    f'<strong>📐 Decisión VIF:</strong> {len(features_revisar_vif)} feature/s con VIF > {VIF_GRAVE} '
    'se mantienen en gradient boosting. En modelos lineales se evaluará en f5_m02.</p>'
    '<p style="color:#4a5568;font-size:14px;margin:0;">'
    f'<strong>📦 Fase 5 recibirá:</strong> {len(features_fase5_todas)} features '
    f'({len(features_seleccionadas)} sin restricción + {len(features_revisar_vif)} con nota VIF). '
    f'Las {len(features_cuarentena)} en cuarentena quedan documentadas pero excluidas.</p>'
    '</div>'
)

# ── Cuerpo completo ───────────────────────────────────────────────────────────
cuerpo = (
    kpis_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:28px 0 12px;">'
    'Comparativa de rangos — estadística vs AutoML</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:16px;">'
    'Azul = rango estadístico (M06: PB+MI). Naranja = rango AutoML. '
    'Verde = puntuación final. Barras cortas = feature relevante en esa fuente.</p>'
    + IMG_COMPARATIVO
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    'Acuerdo entre fuentes</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:16px;">'
    'Zona verde = ambas fuentes coinciden en relevancia. '
    'Puntos fuera de la diagonal = discrepancia que merece análisis en Fase 6.</p>'
    + IMG_SCATTER
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    'Tabla de decisión unificada</h2>'
    '<p style="color:#4a5568;font-size:14px;margin-bottom:8px;">'
    'Ordenada por puntuación final. La columna <strong>Decisión</strong> '
    'indica qué ocurre con cada feature en Fase 5.</p>'
    + guia_tabla_html
    + tabla_decision_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 12px;">'
    'Features por grupo de decisión</h2>'
    + bloque_listas
    + nota_metodologica_html
    + bloque_conclusiones
)

# ── Exportar HTML ─────────────────────────────────────────────────────────────
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m07'
)

html_final = render_base_html(
    titulo='F4·M07 — Selección de Features',
    icono='🎯',
    subtitulo='Fase 4 · EDA Final · Módulo 7',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=cuerpo,
    notebook_nombre='f4_m07_seleccion_features.ipynb',
    notebook_carpeta='fase4_eda'
)

ruta_html = RUTA_FASE4_HTML / 'm07_seleccion_features.html'
guardar_html(html_final, ruta_html)

print('=' * 60)
print('COMPLETADO: F4-M07 Selección de Features')
print('=' * 60)
print(f'  HTML   : {ruta_html}')
print(f'  Parquet: {RUTA_PARQUET}')
print(f'  JSON   : {RUTA_JSON}')
print()
print(f'  Features a Fase 5: {len(features_fase5_todas)} '
      f'({len(features_seleccionadas)} seleccionadas + {len(features_revisar_vif)} revisar VIF)')
print(f'  Cuarentena       : {len(features_cuarentena)}')
print()
print('Siguiente: f4_m08_perfiles_riesgo.ipynb')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m07_seleccion_features.html
COMPLETADO: F4-M07 Selección de Features
  HTML   : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m07_seleccion_features.html
  Parquet: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m07_features_seleccionadas.parquet
  JSON   : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\f4_m07_listas_features.json

  Features a Fase 5: 16 (16 seleccionadas + 0 revisar VIF)
  Cuarentena       : 0

Siguiente: f4_m08_perfiles_riesgo.ipynb
